In [ ]:
%pip install -U torch torchvision transformers datasets evaluate accelerate scikit-learn emoji peft bitsandbytes

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

import evaluate
import numpy as np
from transformers import DataCollatorWithPadding

import re

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
temp = load_dataset("csv", data_files="/content/drive/MyDrive/Copy of Sentiment140.csv", encoding="ISO-8859-1")["train"]
temp = temp.remove_columns(["ID", "Date", "Flag", "User"])
temp = temp.filter(lambda x: x["Label"] in [0, 4])

def relabel(batch):
    labels = batch["Label"]
    new_labels = [0 if l == 0 else 1 if l == 4 else None for l in labels]
    if any(l is None for l in new_labels):
        raise ValueError("Unexpected label in batch")
    return {"Label": new_labels}

def replace(text):
    text = re.sub(r'@\w+', '@USER', text)
    text = re.sub(r'http\S+|www\.\S+', 'HTTPURL', text)
    return text

def clean_text(batch):
    return {"Text": [replace(t) for t in batch["Text"]] }

temp = temp.map(relabel, batched=True)
temp = temp.map(clean_text, batched=True)
temp = temp.train_test_split(test_size=0.1, seed=1)
test_valid = temp["test"].train_test_split(test_size=0.5, seed=1)

#EXPERIMENTAL SAMPLE
dataset_dict = {
    "train": temp["train"].shuffle(seed=1), # EXPERIMENTAL SAMPLE
    "test": test_valid["test"],
    "validation": test_valid["train"],
}

dataset_dict

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
model_path = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(model_path)

id2label = {0: "negative", 1: "positive"}
label2id = {"negative": 0, "positive": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=2, id2label=id2label, label2id=label2id)

In [ ]:
def preprocess_function(x):
    return tokenizer(x["Text"], truncation=True, padding=False, max_length=128)

tokenized = {}
for split, ds in dataset_dict.items():
    ds = ds.rename_column("Label", "labels")
    tokenized[split] = ds.map(preprocess_function, batched=True)

In [ ]:
for n, m in model.named_modules():
    if any(k in n.lower() for k in ("q_proj","v_proj","query","key","value","dense","output","proj","attn")):
        print(n)

from peft import prepare_model_for_kbit_training
model = prepare_model_for_kbit_training(model)

In [ ]:
from peft import LoraConfig, get_peft_model
lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
#import torch
#print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
import os

# Persist checkpoints on Drive (recommended on Colab so you can resume after a runtime restart).
OUTPUT_DIR = "/content/drive/MyDrive/outputs/lora-bertweet"

# Set this to a specific checkpoint folder to resume, or "" to train from scratch.
# Example: "/content/drive/MyDrive/outputs/lora-bertweet/checkpoint-45000"
RESUME_CHECKPOINT = ""

resume_from_checkpoint = (
    RESUME_CHECKPOINT if (RESUME_CHECKPOINT and os.path.isdir(RESUME_CHECKPOINT)) else None
 )

if resume_from_checkpoint:
    print(f"Resuming from checkpoint: {resume_from_checkpoint}")
else:
    print("No checkpoint found (or not set). Training from scratch.")

In [ ]:
from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from transformers.utils.notebook import NotebookProgressCallback
import evaluate, numpy as np

# Fall back to local paths if the resume-config cell wasn't run
try:
    _output_dir = OUTPUT_DIR  # type: ignore[name-defined]
except Exception:
    _output_dir = "outputs/lora-bertweet"

try:
    _resume_from_checkpoint = resume_from_checkpoint  # type: ignore[name-defined]
except Exception:
    _resume_from_checkpoint = None

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir=_output_dir,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    report_to="none",   # important: disables integration callbacks that often trigger this
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
 )

# Remove notebook progress callback (common source of this exact error on Colab)
trainer.remove_callback(NotebookProgressCallback)

trainer.train(resume_from_checkpoint=_resume_from_checkpoint)
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results["eval_accuracy"])
print("Test F1:", results["eval_f1"])

In [ ]:
# Resume training from the latest checkpoint (if any), then evaluate on test
from pathlib import Path
import re

from transformers import DataCollatorWithPadding, TrainingArguments, Trainer
from transformers.utils.notebook import NotebookProgressCallback
import evaluate, numpy as np

# Resolve output_dir: prefer existing TrainingArguments if present
try:
    _output_dir = training_args.output_dir  # type: ignore[name-defined]
except Exception:
    _output_dir = "outputs/lora-bertweet"

output_dir = Path(_output_dir)
checkpoint_dirs = [p for p in output_dir.glob("checkpoint-*") if p.is_dir()]

def _ckpt_step(p: Path) -> int:
    m = re.search(r"checkpoint-(\d+)$", p.name)
    return int(m.group(1)) if m else -1

latest_checkpoint = None
if checkpoint_dirs:
    # Prefer highest step number; fall back to modified time if names are unexpected
    checkpoint_dirs_sorted = sorted(checkpoint_dirs, key=lambda p: (_ckpt_step(p), p.stat().st_mtime))
    latest_checkpoint = str(checkpoint_dirs_sorted[-1])

print(f"Output dir: {output_dir}")
print(f"Latest checkpoint: {latest_checkpoint or 'None (will train from scratch)'}")

# Recreate Trainer (so this cell is runnable after a kernel restart, assuming prior setup cells were run)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(
    output_dir=str(output_dir),
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_steps=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    report_to="none",
    # Optional: uncomment to avoid partial overwrite when resuming with changed args
    # overwrite_output_dir=False,
 )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
 )

# Remove notebook progress callback (it can cause notebook state bugs in some environments)
try:
    trainer.remove_callback(NotebookProgressCallback)
except Exception:
    pass

# Resume if possible; otherwise start fresh
train_result = trainer.train(resume_from_checkpoint=latest_checkpoint) if latest_checkpoint else trainer.train()
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results.get("eval_accuracy"))
print("Test F1:", results.get("eval_f1"))

In [ ]:
model.save_pretrained('/content/drive/MyDrive/outputs/lora-bertweet/final')
tokenizer.save_pretrained('/content/drive/MyDrive/outputs/lora-bertweet/final')

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
from transformers.utils.notebook import NotebookProgressCallback
from peft import PeftConfig, PeftModel
import evaluate
import numpy as np

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftConfig, PeftModel

model_dir = "/content/drive/MyDrive/outputs/lora-bertweet/final"

# Read adapter config to find correct base model (if using PEFT/LoRA)
peft_config = PeftConfig.from_pretrained(model_dir)

tokenizer = AutoTokenizer.from_pretrained(model_dir)  # or peft_config.base_model_name_or_path

base_model = AutoModelForSequenceClassification.from_pretrained(
    peft_config.base_model_name_or_path,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
)

model = PeftModel.from_pretrained(base_model, model_dir)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1": f1.compute(predictions=preds, references=labels, average="binary")["f1"],
    }

eval_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/outputs/eval-lora-bertweet",
    per_device_eval_batch_size=64,
    report_to="none",   # avoid callback integration issues
    do_train=False,
    do_eval=True,
)

trainer = Trainer(
    model=model,
    args=eval_args,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# Avoid notebook callback state bug in Colab
trainer.remove_callback(NotebookProgressCallback)

results = trainer.evaluate(eval_dataset=tokenized["test"])
print("Test accuracy:", results.get("eval_accuracy"))
print("Test F1:", results.get("eval_f1"))